In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU
df_titanic = sns.load_dataset('titanic')

# --- Xử lý dữ liệu thiếu (Missing values) ---
# Điền trung vị cho cột age
df_titanic['age'] = df_titanic['age'].fillna(df_titanic['age'].median())

# Điền giá trị xuất hiện nhiều nhất (mode) cho embark_town
df_titanic['embark_town'] = df_titanic['embark_town'].fillna(df_titanic['embark_town'].mode()[0])

# --- Xoá các cột không cần thiết hoặc trùng lặp ---
# Xoá 'deck' vì thiếu quá nhiều.
# 'alive' trùng 'survived', 'class' trùng 'pclass', 'embarked' trùng 'embark_town'
# 'who', 'adult_male', 'alone' được suy ra từ giới tính và tuổi nên xoá để tránh nhiễu mô hình.
cols_to_drop = ['deck', 'alive', 'class', 'embarked', 'who', 'adult_male', 'alone']
df_titanic = df_titanic.drop(columns=cols_to_drop)

# --- Mã hoá dữ liệu (Encoding) ---
# Đổi giới tính thành 0 và 1
df_titanic['sex'] = df_titanic['sex'].map({'male': 0, 'female': 1})

# One-hot encoding cho embark_town (tạo ra các cột 0/1 cho từng thị trấn, bỏ cột đầu tiên để tránh đa cộng tuyến)
df_titanic = pd.get_dummies(df_titanic, columns=['embark_town'], drop_first=True, dtype=int)

# --- Thiết lập X và y ---
X = df_titanic.drop(columns=['survived'])
y = df_titanic['survived']

# Chia tập dữ liệu (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 2. HUẤN LUYỆN MÔ HÌNH LOGISTIC REGRESSION
# Tăng max_iter lên 1000 để đảm bảo thuật toán hội tụ khi có nhiều features hơn
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("--- KẾT QUẢ LOGISTIC REGRESSION ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_log):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_log):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_log):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_log):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))
print("\n")


# 3. HUẤN LUYỆN LINEAR REGRESSION
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin_continuous = lin_model.predict(X_test)
y_pred_lin_class = (y_pred_lin_continuous >= 0.5).astype(int)

print("--- KẾT QUẢ LINEAR REGRESSION (Ép ngưỡng 0.5) ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lin_class):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lin_class):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_lin_class):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_lin_class):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lin_class))

--- KẾT QUẢ LOGISTIC REGRESSION ---
Accuracy : 0.8101
Precision: 0.7857
Recall   : 0.7432
F1-score : 0.7639
Confusion Matrix:
[[90 15]
 [19 55]]


--- KẾT QUẢ LINEAR REGRESSION (Ép ngưỡng 0.5) ---
Accuracy : 0.7933
Precision: 0.7681
Recall   : 0.7162
F1-score : 0.7413
Confusion Matrix:
[[89 16]
 [21 53]]


**Nhận xét so sánh Linear Regression và Logistic Regression:**

Khả năng giải quyết bài toán phân loại: * Logistic Regression sinh ra là để làm bài toán phân loại. Đầu ra của nó được đi qua hàm Sigmoid, giới hạn giá trị từ 0 đến 1, đại diện cho xác suất sống sót rất hợp lý.

Linear Regression sinh ra để dự đoán các con số liên tục (ví dụ: giá nhà, giá vé). Khi ép nó dự đoán biến mục tiêu survived (0 và 1), đường thẳng dự đoán của nó có thể trả ra các giá trị vô lý như -0.2 (xác suất âm) hoặc 1.5 (xác suất > 100%). Việc chúng ta phải "ép" một ngưỡng 0.5 thủ công để tính điểm phân loại cho Linear Regression là một sự gượng ép về mặt toán học.

Chỉ số đánh giá (Metrics): * Dù trong một số trường hợp, điểm Accuracy của Linear Regression (sau khi ép ngưỡng) có thể gần bằng Logistic Regression, nhưng nó rất dễ bị đánh lừa bởi các điểm dữ liệu ngoại lệ (outliers).

Confusion Matrix của Logistic Regression thường cân bằng và bắt được lớp thiểu số (người sống sót) tốt hơn nhờ tối ưu hóa bằng hàm Loss (Log-Loss) thay vì MSE như Linear.

Kết luận: Đối với bài toán phân loại hành khách (Sống/Chết), Logistic Regression là mô hình chuẩn xác, khoa học và phù hợp hơn hoàn toàn so với Linear Regression.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. ĐỌC TRỰC TIẾP TỪ 2 FILE CSV ĐÃ CHIA SẴN
try:
    df_train = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
    df_test = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')
    print("Đã đọc thành công 2 file CSV!")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy thư mục 'Dry_Bean_Dataset'.")

if 'Unnamed: 0' in df_train.columns:
    df_train = df_train.drop(columns=['Unnamed: 0'])
    df_test = df_test.drop(columns=['Unnamed: 0'])

X_train = df_train.drop(columns=['class'])
y_train = df_train['class']

X_test = df_test.drop(columns=['class'])
y_test = df_test['class']

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. MÔ HÌNH 1: LOGISTIC REGRESSION
log_bean = LogisticRegression(max_iter=2000)
log_bean.fit(X_train_scaled, y_train_encoded)
y_pred_log = log_bean.predict(X_test_scaled)

print("--- KẾT QUẢ LOGISTIC REGRESSION (DRY BEAN) ---")
print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred_log):.4f}")
print("Classification Report:")
print(classification_report(y_test_encoded, y_pred_log, target_names=le.classes_))

# 3. MÔ HÌNH 2: K-NEAREST NEIGHBORS (KNN)
knn_bean = KNeighborsClassifier(n_neighbors=5)
knn_bean.fit(X_train_scaled, y_train_encoded)
y_pred_knn = knn_bean.predict(X_test_scaled)

print("\n--- KẾT QUẢ K-NEAREST NEIGHBORS (DRY BEAN) ---")
print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred_knn):.4f}")
print("Classification Report:")
print(classification_report(y_test_encoded, y_pred_knn, target_names=le.classes_))

Đã đọc thành công 2 file CSV!
--- KẾT QUẢ LOGISTIC REGRESSION (DRY BEAN) ---
Accuracy: 0.9192
Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709


--- KẾT QUẢ K-NEAREST NEIGHBORS (DRY BEAN) ---
Accuracy: 0.9155
Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.88      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.95  